In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout

In [4]:
EPOCHS = 32
BATCH_SIZE = 256
INPUT_FILE_NAME = '../datasets/frankenstein.txt'
WINDOW_LENGTH = 40
WINDOW_STEP = 3

In [5]:
# Open the input file.
file = open(INPUT_FILE_NAME, 'r', encoding='utf-8')
text = file.read()
file.close()
# Make lowercase and remove newline and extra spaces.
text = text.lower()
text = text.replace('\n', ' ')
text = text.replace(' ', ' ')

In [6]:
len(text)

457713

In [7]:
# Encode characters as indices.
unique_chars = list(set(text))
char_to_index = dict((ch, index) for index, ch in enumerate(unique_chars))
index_to_char = dict((index, ch) for index, ch in enumerate(unique_chars))

encoding_width = len(char_to_index)

In [8]:
encoding_width

70

In [9]:
# Create training examples.
fragments = []
targets = []

for i in range(0, len(text) - WINDOW_LENGTH, WINDOW_STEP):
    fragments.append(text[i: i + WINDOW_LENGTH])
    targets.append(text[i + WINDOW_LENGTH])

In [10]:
len(fragments)

152558

In [11]:
# Convert to one-hot encoded training data.
X = np.zeros((len(fragments), WINDOW_LENGTH, encoding_width))
y = np.zeros((len(fragments), encoding_width))
for i, fragment in enumerate(fragments):
    for j, char in enumerate(fragment):
        X[i, j, char_to_index[char]] = 1
    target_char = targets[i]
    y[i, char_to_index[target_char]] = 1

In [ ]:
# Build and train model.
model = Sequential()
model.add(LSTM(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.2,input_shape=(None, encoding_width)))
model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(encoding_width, activation='softmax'))
model.compile(loss='categorical_crossentropy',optimizer='adam')
model.summary()
history = model.fit(X, y, validation_split=0.05,batch_size=BATCH_SIZE,epochs=EPOCHS, verbose=2,shuffle=True)

/Users/shubham.kumar.3/Desktop/Deep-Learning/.venv/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, None, 128)      │       101,888 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 70)             │         9,030 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 242,502 (947.27 KB)

 Trainable params: 242,502 (947.27 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/32
567/567 - 83s - 147ms/step - loss: 2.8138 - val_loss: 2.6742
Epoch 2/32
567/567 - 84s - 148ms/step - loss: 2.4167 - val_loss: 2.5131
Epoch 3/32
567/567 - 84s - 148ms/step - loss: 2.2770 - val_loss: 2.4046
Epoch 4/32
567/567 - 83s - 147ms/step - loss: 2.1829 - val_loss: 2.3268
Epoch 5/32
567/567 - 82s - 145ms/step - loss: 2.1130 - val_loss: 2.2836
Epoch 6/32


In [ ]:
letters = 'the boy '
one_hots = []
for i, char in enumerate(letters):
    x = np.zeros(encoding_width)
    x[char_to_index[char]] = 1
    one_hots.append(x)

new = np.array(one_hots).reshape(1, len(letters), encoding_width)

In [29]:
new.shape

(1, 8, 60)

In [30]:
y_predict = model.predict(new, verbose=1)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step


In [31]:
y_predict

array([[2.17479965e-06, 1.71712916e-02, 3.25481708e-09, 5.53453192e-02,
        1.84156734e-09, 4.66284873e-05, 3.44867110e-02, 8.51005837e-02,
        4.17388836e-03, 3.72619361e-05, 1.05369538e-04, 1.89857983e-05,
        2.53964028e-09, 1.16515495e-02, 1.69609971e-02, 5.16715133e-08,
        3.09084989e-02, 1.00095123e-08, 5.23357987e-02, 1.51717728e-07,
        6.43259318e-06, 5.77265695e-02, 1.36812654e-04, 6.17907903e-09,
        3.07076220e-07, 4.92304389e-05, 2.93568301e-05, 9.09808223e-05,
        3.17771587e-09, 2.71451585e-02, 1.41079187e-01, 4.99627180e-02,
        7.11073014e-08, 3.56815271e-08, 3.77097502e-02, 3.47927725e-03,
        4.13865522e-02, 9.29105096e-04, 3.12288057e-06, 4.11236215e-06,
        5.04293777e-02, 5.11450189e-08, 2.11549134e-09, 2.18455301e-04,
        1.22932752e-03, 3.06810143e-05, 2.79013566e-05, 5.93126879e-06,
        1.33288893e-06, 5.15656825e-03, 1.53346872e-02, 3.19419317e-02,
        1.39927433e-04, 6.45333103e-06, 8.42363193e-07, 4.907618

In [32]:
np.argmax(y_predict)

np.int64(30)

In [33]:
index_to_char[np.argmax(y_predict)]

'a'

In [ ]:
# Number of characters to generate
NUM_PREDICTIONS = 12

# Seed text
seed_text = "the boy who "
generated = seed_text

# Prepare one-hot input from the seed text
one_hots = []
for char in seed_text:
    x = np.zeros(encoding_width)
    x[char_to_index[char]] = 1
    one_hots.append(x)

input_seq = np.array(one_hots).reshape(1, len(seed_text), encoding_width)

# Generate next NUM_PREDICTIONS characters
for i in range(NUM_PREDICTIONS):
    # Predict probabilities for next char
    y_predict = model.predict(input_seq, verbose=0)
    next_index = np.argmax(y_predict)
    next_char = index_to_char[next_index]

    print(f"Predicted letter {i+1}: {next_char}")
    generated += next_char

    # Update input sequence: append new char & drop oldest if longer than WINDOW_LENGTH
    next_one_hot = np.zeros((1, 1, encoding_width))
    next_one_hot[0, 0, next_index] = 1
    input_seq = np.concatenate([input_seq, next_one_hot], axis=1)

    if input_seq.shape[1] > WINDOW_LENGTH:
        input_seq = input_seq[:, 1:, :]   # keep only last WINDOW_LENGTH chars

print("\nGenerated text:\n", generated)

Predicted letter 1: h
Predicted letter 2: a
Predicted letter 3: d
Predicted letter 4:  
Predicted letter 5: b
Predicted letter 6: e
Predicted letter 7: e
Predicted letter 8: n
Predicted letter 9:  
Predicted letter 10: t
Predicted letter 11: h
Predicted letter 12: e

Generated text:
 the boy who had been the


In [ ]:
## save model for reuse
model.save("language_model_LSTM.keras")

In [36]:
## load saved model
loaded_model = keras.models.load_model("language_model_LSTM.keras")

In [19]:
## predict using saved model
y_test = loaded_model(new)

In [20]:
## convert index to character
index_to_char[np.argmax(y_test)]

't'

In [ ]:
# Number of characters to generate
NUM_PREDICTIONS = 15
encoding_width=60

# Seed text
seed_text = "the gulf "
generated = seed_text

# Prepare one-hot input from the seed text
one_hots = []
for char in seed_text:
    x = np.zeros(encoding_width)
    x[char_to_index[char]] = 1
    one_hots.append(x)

input_seq = np.array(one_hots).reshape(1, len(seed_text), encoding_width)

# Generate next NUM_PREDICTIONS characters
for i in range(NUM_PREDICTIONS):
    # Predict probabilities for next char
    y_predict = loaded_model.predict(input_seq, verbose=0)
    next_index = np.argmax(y_predict)
    next_char = index_to_char[next_index]

    print(f"Predicted letter {i+1}: {next_char}")
    generated += next_char

    # Update input sequence: append new char & drop oldest if longer than WINDOW_LENGTH
    next_one_hot = np.zeros((1, 1, encoding_width))
    next_one_hot[0, 0, next_index] = 1
    input_seq = np.concatenate([input_seq, next_one_hot], axis=1)

    if input_seq.shape[1] > WINDOW_LENGTH:
        input_seq = input_seq[:, 1:, :]   # keep only last WINDOW_LENGTH chars

print("\nGenerated text:\n", generated)

Predicted letter 1: a
Predicted letter 2: n
Predicted letter 3: d
Predicted letter 4:  
Predicted letter 5: t
Predicted letter 6: h
Predicted letter 7: e
Predicted letter 8:  
Predicted letter 9: s
Predicted letter 10: e
Predicted letter 11: e
Predicted letter 12: t
Predicted letter 13:  
Predicted letter 14: o
Predicted letter 15: f

Generated text:
 the gulf and the seet of
